In [63]:
%pip install pandas numpy scikit-learn
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [64]:
import pandas as pd
import numpy as np
import pickle
import warnings
from datetime import datetime
from sklearn.preprocessing import RobustScaler

import matplotlib.pyplot as plt
import pickle


warnings.filterwarnings('ignore')

In [65]:
# STEP 1: LOAD THE DATA

df = pd.read_csv('energy_dataset.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').reset_index(drop=True)

In [66]:
df

,time,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,generation hydro pumped storage aggregated,generation hydro pumped storage consumption,generation hydro run-of-river and poundage,generation hydro water reservoir,generation marine,generation nuclear,generation other,generation other renewable,generation solar,generation waste,generation wind offshore,generation wind onshore,forecast solar day ahead,forecast wind offshore eday ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual
0,2014-12-31 23:00:00+00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,NaN,863.0,1051.0,1899.0,0.0,7096.0,43.0,73.0,49.0,196.0,0.0,6378.0,17.0,NaN,6436.0,26118.0,25385.0,50.10,65.41
1,2015-01-01 00:00:00+00:00,449.0,328.0,0.0,5196.0,4755.0,158.0,0.0,0.0,0.0,NaN,920.0,1009.0,1658.0,0.0,7096.0,43.0,71.0,50.0,195.0,0.0,5890.0,16.0,NaN,5856.0,24934.0,24382.0,48.10,64.92
2,2015-01-01 01:00:00+00:00,448.0,323.0,0.0,4857.0,4581.0,157.0,0.0,0.0,0.0,NaN,1164.0,973.0,1371.0,0.0,7099.0,43.0,73.0,50.0,196.0,0.0,5461.0,8.0,NaN,5454.0,23515.0,22734.0,47.33,64.48
3,2015-01-01 02:00:00+00:00,438.0,254.0,0.0,4314.0,4131.0,160.0,0.0,0.0,0.0,NaN,1503.0,949.0,779.0,0.0,7098.0,43.0,75.0,50.0,191.0,0.0,5238.0,2.0,NaN,5151.0,22642.0,21286.0,42.27,59.32
4,2015-01-01 03:00:00+00:00,428.0,187.0,0.0,4130.0,3840.0,156.0,0.0,0.0,0.0,NaN,1826.0,953.0,720.0,0.0,7097.0,43.0,74.0,42.0,189.0,0.0,4935.0,9.0,NaN,4861.0,21785.0,20264.0,38.41,56.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,2018-12-31 18:00:00+00:00,297.0,0.0,0.0,7634.0,2628.0,178.0,0.0,0.0,0.0,NaN,1.0,1135.0,4836.0,0.0,6073.0,63.0,95.0,85.0,277.0,0.0,3113.0,96.0,NaN,3253.0,30619.0,30653.0,68.85,77.02
35060,2018-12-31 19:00:00+00:00,296.0,0.0,0.0,7241.0,2566.0,174.0,0.0,0.0,0.0,NaN,1.0,1172.0,3931.0,0.0,6074.0,62.0,95.0,33.0,280.0,0.0,3288.0,51.0,NaN,3353.0,29932.0,29735.0,68.40,76.16
35061,2018-12-31 20:00:00+00:00,292.0,0.0,0.0,7025.0,2422.0,168.0,0.0,0.0,0.0,NaN,50.0,1148.0,2831.0,0.0,6076.0,61.0,94.0,31.0,286.0,0.0,3503.0,36.0,NaN,3404.0,27903.0,28071.0,66.88,74.30
35062,2018-12-31 21:00:00+00:00,293.0,0.0,0.0,6562.0,2293.0,163.0,0.0,0.0,0.0,NaN,108.0,1128.0,2068.0,0.0,6075.0,61.0,93.0,31.0,287.0,0.0,3586.0,29.0,NaN,3273.0,25450.0,25801.0,63.93,69.89


In [67]:
# STEP 2: DROP USELESS COLUMNS

columns_to_drop = [
    'generation_hydro_pumped_storage_aggregated',
    'generation_geothermal',
    'generation_marine',
    'forecast_wind_offshore_day_ahead'
]

df = df.drop(columns=columns_to_drop, errors='ignore')

In [68]:
# STEP 3: CREATE TIME FEATURES (FIXED!)

df['hour'] = df['time'].dt.hour                    # 0-23
df['day_of_week'] = df['time'].dt.dayofweek       # 0-6
df['month'] = df['time'].dt.month                 # 1-12
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# DON'T DROP HOUR - You use it in STEP 10!

# Weekend flag
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Holiday
df['is_holiday'] = 0


In [69]:
# After: df = pd.read_csv('electricity_data.csv')

print("Column names in your CSV:")
print(df.columns.tolist())

Column names in your CSV:
['time', 'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil coal-derived gas', 'generation fossil gas', 'generation fossil hard coal', 'generation fossil oil', 'generation fossil oil shale', 'generation fossil peat', 'generation geothermal', 'generation hydro pumped storage aggregated', 'generation hydro pumped storage consumption', 'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation marine', 'generation nuclear', 'generation other', 'generation other renewable', 'generation solar', 'generation waste', 'generation wind offshore', 'generation wind onshore', 'forecast solar day ahead', 'forecast wind offshore eday ahead', 'forecast wind onshore day ahead', 'total load forecast', 'total load actual', 'price day ahead', 'price actual', 'hour', 'day_of_week', 'month', 'hour_sin', 'hour_cos', 'is_weekend', 'is_holiday']


In [70]:
# STEP 4: CREATE LAG FEATURES (Past Values)

# Past demand values
df['demand_lag_1h'] = df['total load actual'].shift(1)      # 1 hour ago
df['demand_lag_24h'] = df['total load actual'].shift(24)    # Yesterday same hour
df['demand_lag_168h'] = df['total load actual'].shift(168)  # Last week

# Past price values
df['price_lag_1h'] = df['price actual'].shift(1)            # 1 hour ago
df['price_lag_24h'] = df['price actual'].shift(24)          # Yesterday

In [71]:
# STEP 5: CREATE GENERATION FEATURES

print("\n[STEP 5] Creating generation features...")

# Add all renewable sources together
df['renewable'] = (
    df['generation solar'].fillna(0) +
    df['generation wind onshore'].fillna(0) +
    df['generation wind offshore'].fillna(0) +
    df['generation hydro run-of-river and poundage'].fillna(0) +
    df['generation hydro water reservoir'].fillna(0)
)

# Add all fossil sources together
df['fossil'] = (
    df['generation fossil gas'].fillna(0) +
    df['generation fossil hard coal'].fillna(0) +
    df['generation fossil brown coal/lignite'].fillna(0)
)

# Nuclear and biomass
df['nuclear'] = df['generation nuclear'].fillna(0)
df['biomass'] = df['generation biomass'].fillna(0)

# Total generation
df['total_gen'] = (
    df['renewable'] +
    df['fossil'] +
    df['nuclear'] +
    df['biomass']
)




[STEP 5] Creating generation features...


In [72]:
# STEP 6: CREATE  RATIOS

df['renewable_pct'] = np.where(
    df['total_gen'] > 0,
    (df['renewable'] / df['total_gen']) * 100,
    0
)

In [73]:
# STEP 7: CREATE ROLLING AVERAGES


# Average demand over last 24 hours
df['demand_avg_24h'] = df['total load actual'].rolling(window=24, min_periods=1).mean()

# Average price over last 24 hours
df['price_avg_24h'] = df['price actual'].rolling(window=24, min_periods=1).mean()


In [74]:
# STEP 8: REMOVE EMPTY ROWS

# From lag features, first few rows will be empty
rows_before = len(df)
df = df.dropna(subset=['demand_lag_168h', 'price_lag_24h'])  # Only drop rows missing lag features
rows_after = len(df)
rows_removed = rows_before - rows_after

In [75]:
# STEP 9: HANDLE OUTLIERS (ROBUST METHOD)

scaler_demand = RobustScaler()
scaler_price = RobustScaler()

# Fit on training data ONLY
split_point = int(len(df) * 0.8)

df['demand scaled'] = scaler_demand.fit_transform(
    df[['total load actual']]
)

df['price scaled'] = scaler_price.fit_transform(
    df[['price actual']]
)


In [76]:
# STEP 10: SELECT FEATURES FOR MODELS

# Features to use for predicting DEMAND
demand_features = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday',
    'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
    'price_lag_1h', 'price_lag_24h',
    'renewable', 'fossil', 'nuclear',
    'renewable_pct',
    'demand_avg_24h', 'price_avg_24h'
]

# Features to use for predicting PRICE
price_features = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday',
    'price_lag_1h', 'price_lag_24h',
    'demand_lag_1h', 'demand_lag_24h',
    'renewable', 'fossil', 'nuclear',
    'renewable_pct',
    'price_avg_24h', 'demand_avg_24h'
]

In [78]:
df.head(55)

,time,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,generation hydro pumped storage aggregated,generation hydro pumped storage consumption,generation hydro run-of-river and poundage,generation hydro water reservoir,generation marine,generation nuclear,generation other,generation other renewable,generation solar,generation waste,generation wind offshore,generation wind onshore,forecast solar day ahead,forecast wind offshore eday ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual,hour,day_of_week,month,hour_sin,hour_cos,is_weekend,is_holiday,demand_lag_1h,demand_lag_24h,demand_lag_168h,price_lag_1h,price_lag_24h,renewable,fossil,nuclear,biomass,total_gen,renewable_pct,demand_avg_24h,price_avg_24h,demand scaled,price scaled
168,2015-01-07 23:00:00+00:00,546.0,571.0,0.0,4178.0,7280.0,383.0,0.0,0.0,0.0,NaN,398.0,658.0,831.0,0.0,6741.0,84.0,67.0,433.0,260.0,0.0,5908.0,443.0,NaN,5755.0,27766.0,26788.0,49.00,73.73,23,2,1,-2.588190e-01,9.659258e-01,0,0,30477.0,30518.0,25385.0,75.07,67.24,7830.0,12029.0,6741.0,546.0,27146.0,28.844029,31949.583333,79.046250,-0.288151,0.845040
169,2015-01-08 00:00:00+00:00,516.0,566.0,0.0,3912.0,6774.0,370.0,0.0,0.0,0.0,NaN,392.0,628.0,942.0,0.0,6741.0,85.0,67.0,391.0,257.0,0.0,5509.0,363.0,NaN,5596.0,26176.0,25146.0,46.86,70.99,0,3,1,0.000000e+00,1.000000e+00,0,0,26788.0,28484.0,24382.0,73.73,64.17,7470.0,11252.0,6741.0,516.0,25979.0,28.753994,31810.500000,79.330417,-0.510441,0.698123
170,2015-01-08 01:00:00+00:00,508.0,455.0,0.0,3718.0,6349.0,372.0,0.0,0.0,0.0,NaN,956.0,631.0,882.0,0.0,6742.0,85.0,67.0,352.0,256.0,0.0,5204.0,295.0,NaN,5218.0,24974.0,23889.0,45.68,68.30,1,3,1,2.588190e-01,9.659258e-01,0,0,25146.0,27026.0,22734.0,70.99,62.12,7069.0,10522.0,6742.0,508.0,24841.0,28.456986,31679.791667,79.587917,-0.680611,0.553887
171,2015-01-08 02:00:00+00:00,509.0,369.0,0.0,3768.0,6078.0,373.0,0.0,0.0,0.0,NaN,1088.0,634.0,934.0,0.0,6743.0,86.0,65.0,328.0,251.0,0.0,4891.0,276.0,NaN,4923.0,24173.0,23046.0,44.99,64.22,2,3,1,5.000000e-01,8.660254e-01,0,0,23889.0,26248.0,21286.0,68.30,62.11,6787.0,10215.0,6743.0,509.0,24254.0,27.983013,31546.375000,79.675833,-0.794734,0.335121
172,2015-01-08 03:00:00+00:00,518.0,367.0,0.0,3707.0,5984.0,373.0,0.0,0.0,0.0,NaN,1027.0,636.0,799.0,0.0,6744.0,86.0,67.0,327.0,251.0,0.0,4738.0,244.0,NaN,4706.0,23779.0,22587.0,44.81,63.53,3,3,1,7.071068e-01,7.071068e-01,0,0,23046.0,25838.0,20264.0,64.22,60.05,6500.0,10058.0,6744.0,518.0,23820.0,27.287993,31410.916667,79.820833,-0.856872,0.298123
173,2015-01-08 04:00:00+00:00,528.0,351.0,0.0,3706.0,5817.0,361.0,0.0,0.0,0.0,NaN,1348.0,631.0,781.0,0.0,6748.0,86.0,66.0,320.0,249.0,0.0,4868.0,212.0,NaN,4779.0,23503.0,22361.0,44.68,66.70,4,3,1,8.660254e-01,5.000000e-01,0,0,22587.0,26021.0,19905.0,63.53,62.48,6600.0,9874.0,6748.0,528.0,23750.0,27.789474,31258.416667,79.996667,-0.887467,0.468097
174,2015-01-08 05:00:00+00:00,533.0,367.0,0.0,3946.0,5785.0,354.0,0.0,0.0,0.0,NaN,1804.0,631.0,943.0,0.0,6750.0,86.0,67.0,265.0,246.0,0.0,4809.0,235.0,NaN,4867.0,23493.0,22551.0,44.81,69.79,5,3,1,9.659258e-01,2.588190e-01,0,0,22361.0,26992.0,20010.0,66.70,70.01,6648.0,10098.0,6750.0,533.0,24029.0,27.666570,31073.375000,79.987500,-0.861746,0.633780
175,2015-01-08 06:00:00+00:00,525.0,360.0,0.0,3890.0,5570.0,347.0,0.0,0.0,0.0,NaN,1638.0,629.0,888.0,0.0,6748.0,86.0,66.0,223.0,246.0,0.0,4849.0,420.0,NaN,4812.0,23562.0,22543.0,44.81,75.57,6,3,1,1.000000e+00,6.123234e-17,0,0,22551.0,28625.0,20377.0,69.79,75.20,6589.0,9820.0,6748.0,525.0,23682.0,27.822819,30819.958333,80.002917,-0.862829,0.943700
176,2015-01-08 07:00:00+00:00,529.0,457.0,0.0,4066.0,6289.0,332.0,0.0,0.0,0.0,NaN,508.0,641.0,1147.0,0.0,6747.0,86.0,69.0,708.0,216.0,0.0,4285.0,1166.0,NaN,4532.0,24929.0,24042.0,46.85,82.36,7,3,1,9.659258e-01,-2.588190e-01,0,0,